In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    fbeta_score, precision_score, recall_score,
    matthews_corrcoef, average_precision_score, confusion_matrix,
    precision_recall_curve
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")


In [2]:
CSV_PATH = "../data/creditcard.csv"

def load_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    assert {"Time", "Class"} <= set(df.columns), "Colunas Time/Class ausentes"
    n0 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicatas removidas: {n0 - len(df)}")
    return df

def temporal_split(df: pd.DataFrame, val_frac: float = 0.15, test_frac: float = 0.15):
    """Split temporal em treino/validação/teste, ordenado por 'Time'."""
    df = df.sort_values("Time").reset_index(drop=True)
    n = len(df)
    i_val = int(n * (1 - val_frac - test_frac))
    i_test = int(n * (1 - test_frac))
    train_df = df.iloc[:i_val].copy()
    val_df = df.iloc[i_val:i_test].copy()
    test_df = df.iloc[i_test:].copy()

    for nome, d in [("Treino", train_df), ("Validação", val_df), ("Teste", test_df)]:
        print(f"{nome:<10}: {len(d):>7} transações | Fraudes: {int(d['Class'].sum()):>4} "
              f"({d['Class'].mean()*100:.4f}%)")
    return train_df, val_df, test_df

df = load_data(CSV_PATH)
train_df, val_df, test_df = temporal_split(df)

Duplicatas removidas: 1081
Treino    :  198608 transações | Fraudes:  366 (0.1843%)
Validação :   42559 transações | Fraudes:   55 (0.1292%)
Teste     :   42559 transações | Fraudes:   52 (0.1222%)
